# SNN Hackathon 2026 – MNE-Python Preprocessing Tutorial
## Day 2: Bad Channels · ICA · Re-referencing · Epochs · Baseline · Rejection · ERP
 
---
 
At this stage you should have already loaded your raw EEG data, merged broken files, resampled and applied band-pass filtering.
Today we finalise the preprocessing pipeline and create clean, baseline-corrected epochs ready for ERP analysis.
 
**Pipeline overview:**
```
Load filtered data (Day 1 output)
         │
         ▼
   1. Bad Channels            (set montage · interpolate known bad electrodes)
         │
         ▼
   2. ICA on continuous data  (fit on 1 Hz HP copy → apply on real data)
         │
         ▼
   3. Re-referencing          (average reference · butterfly plot)
         │
         ▼
   4. Extract Epochs          (time-lock to stimulus onset)
         │
         ▼
   5. Baseline Correction     (−100 to 0 ms)
         │
         ▼
   6. Epoch Rejection         (amplitude threshold ±150 µV)
         │
         ▼
   7. ERP Preview & Save
```
 
> 💡 **Why this order?** Filtering, re-referencing, and averaging are **linear operations** — their order does not matter mathematically.
> However, **ICA should run before re-referencing**: average referencing spreads artifact activity (e.g. eye blinks) across all channels, which can degrade the ICA decomposition.
> Bad channels must be interpolated **before ICA** so they don't corrupt the decomposition.
> Artifact rejection is **nonlinear** (it uses a threshold). Order matters for nonlinear operations — always reject *after* re-referencing and baseline correction.
> *(Luck, S.J. 2014, An Introduction to the Event-Related Potential Technique; erpinfo.org/order-of-steps)*
 
### How to use this notebook
- Replace every `???` with your own code
- **Solution cells** follow directly below each task cell — try it yourself first!
- Read every markdown cell before writing code — it explains what to do and *why*
 

---
## 0 · Imports & Setup

In [ ]:
import os
import os.path as op
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import ICA, create_eog_epochs
 
mne.set_log_level('WARNING')  # suppress info messages, keep warnings
%matplotlib inline
 
print('MNE version:', mne.__version__)

## 0.1 · Paths & participant
 
We need **one** file path:
 
| Variable | What it is | Used for |
|---|---|---|
| `raw_filt_path` | Band-pass filtered output from Day 1 (0.2–40 Hz) | Everything — including the 1 Hz HP copy for ICA |
 
> 💡 ICA needs a **1 Hz high-pass** version to avoid slow drifts corrupting the decomposition.
> We create that copy from the same preprocessed data (`raw.copy().filter(l_freq=1.0)`) and use it *only* for training — not as our output.
> This keeps the entire pipeline on a **single data chain**, avoiding inconsistencies.

In [ ]:
# Paths & participant ID
DATA_DIR  = r'C:\Your\Path'  # <-- change this!
sub_id    = '???'              # <-- change this!
ORIG_DATA_DIR = ''

# Output from Day 1
raw_filt_path = op.join(DATA_DIR, 'preprocessing', f'{sub_id}_filtered_raw.fif')
data_path     = DATA_DIR

print(f'Participant: {sub_id}')
print(f'File: {raw_filt_path}')


raw = mne.io.read_raw_fif(???, preload=True)
raw_filtered = mne.io.read_raw_fif(???, preload=True)
print(raw.info)
print(raw_filtered.info)

## Part 0.2 Mark Bad Channels
Channels with bad signal quality need to be marked (not removed!) before doing any calculations (like ICA), so they are excluded from the calculations. 
They only need to be marked in the original, unfiltered Data. 

In [ ]:
raw.info['bads'].append(['P10'])    
print(raw.info)

---
## Part 1 · ICA – Artifact Correction on Continuous Data
 
### What is ICA?
 
**Independent Component Analysis** decomposes the mixed EEG signal into a set of statistically
**independent source signals** (components). Artifacts like eye blinks have very characteristic
patterns — large, sharp frontal waves — and ICA isolates these into their own components.
We remove those components and reconstruct the signal from the rest.
 
### Why fit ICA on *continuous* data?
 
ICA needs long, uninterrupted stretches of data to reliably find independent sources.
If you epoch first and then fit ICA, artifacts at epoch boundaries may be split across
two trials and missed.

### Why a 1 Hz high-pass copy for *fitting*?
 
ICA is sensitive to slow signal drifts. Our Day 1 data has a 0.2 Hz lower cutoff —
that still allows some drift through, which can confuse ICA.
 
Best practice is:
```
Unfiltered raw  →  copy  →  1 Hz high-pass  →  ica.fit()    ← only for training
ica.apply()         on  filtered raw (0.2 Hz filtered, interpolated)       ← actual output
```
 
### Why ICA *before* re-referencing?
 
Average re-referencing subtracts the mean of all channels from each channel.
This **spreads** artifact activity (e.g. a blink concentrated at Fp1/Fp2) across
*every* electrode, making it harder for ICA to isolate the artifact into a single component.
Running ICA first — on data that still has the original recording reference — gives
the cleanest decomposition.
 
### How to identify artifact components (ICLabel criteria)
 
Use **three complementary plots** — topographic maps, time series, and correlation scores:
 
| Artifact | Topography | Time series | Power spectrum |
|---|---|---|---|
| Eye blink | Large symmetric loading at Fp1/Fp2 | Regular large spikes | Most power <5 Hz |
| Eye movement | Asymmetric frontal (left/right) | Step-function pattern | Most power <5 Hz |
| Heartbeat | Near-linear gradient | Regular QRS complexes (~1/s) | No clear peaks |
| Muscle | Focal, shallow (scalp edge) | Non-stationary bursts | Broadband >20 Hz |
| Brain (KEEP!) | Dipolar map, RV <15% | ERP if epoched | 1/f + peak 5–30 Hz |
 
*(Based on ICLabel taxonomy — labeling.ucsd.edu/tutorial/labels)*
 
> ⚠️  **When in doubt, keep the component.** Removing a brain component is worse than
> leaving a small artifact residual.

### 📌 Task 1.1 – Prepare the 1 Hz high-pass copy for ICA fitting
 
Create a **copy** of the interpolated raw data and apply a 1 Hz high-pass filter.
This copy is used *only* for `ica.fit()` — never as the final output.

In [ ]:
raw_ica = raw.???(l_freq=???, h_freq=None)
print('1 Hz HP copy ready.')

### 📌 Task 1.2 – Initialise and fit ICA
 
`random_state=100` gives a fixed seed so **everyone gets the same result** (reproducibility).

In [ ]:
ica = ICA(n_components=???, max_iter=???, random_state=???)
ica.fit(???)

### 📌 Task 1.3 – Inspect components visually
 
Use **two plots** to identify artifact components:
 
**1. `plot_components()`** — topographic maps:
- **Eye blinks**: large symmetric loading at Fp1/Fp2
- **Eye movements**: asymmetric frontal pattern
- **Heartbeat**: near-linear gradient across scalp
- **Muscle**: focal, concentrated at scalp edges (shallow source)
- **Brain (keep!)**: clean dipolar map, typically one polarity on each hemisphere
 
**2. `plot_sources()`** — time series:
- **Eye blinks**: unmistakable large, regular spikes
- **Heartbeat**: rhythmic QRS-shaped pulses (~1 per second)
- **Muscle**: high-frequency irregular bursts
 
> 💡 We plot sources on `raw` (filtered + interpolated) so the time scale is meaningful.

In [ ]:
# Topographic maps of all components
ica.plot_components()

# Time-series of each component — plotted on the real (non-1-Hz) data
ica.plot_sources(raw, show_scrollbars=False)

### 📌 Task 1.4 – Find and exclude artifact components
 
Two approaches — combine for best results:
 
**Option A – Manual:** enter component indices you identified from the plots.
 
**Option B – Automatic (`find_bads_eog()`):** correlates each component with the EOG channel.
High correlation = likely eye artifact. Use as a starting point, then verify visually.
 
> 💡 Similar functions: `find_bads_ecg()` for heartbeat, `find_bads_muscle()` for EMG.

In [ ]:
bad_components = [???]   # indices from plot_components()
print(f'Components to remove: {bad_components}')

### 📌 Task 1.5 – Preview: plot_overlay
 
`plot_overlay()` shows a **before/after comparison** at the channel level.
Red = original signal, black = signal after removing the bad components.
 
**Check:** Do the blink spikes disappear? Does the underlying brain signal survive?
The ERP-like shape should remain — only the artifact spikes should disappear.

In [ ]:
ica.plot_overlay(???, exclude=bad_components, picks='eeg')

### 📌 Task 1.6 – Apply ICA to the data
 
**Important:** apply to `raw` (Day 1 filtered + interpolated), **not** to `raw_ica`.
`raw_ica` was only for training the model. We use `.copy()` to leave `raw` intact.

In [ ]:
raw_clean = ica.apply(???, exclude=bad_components)
print(f'Artifacts removed: {bad_components}')

---
## Part 2 · Re-referencing
 
### What is a reference?
 
An EEG electrode does **not** measure absolute voltage — it always measures the **difference**
between itself and a chosen **reference electrode**. The online-reference is set during recording
(e.g. Cz, one mastoid). If that reference electrode sits in a brain-active area, its neural
activity leaks into *all* other channels, creating a systematic bias.
 
### Average reference
 
The standard solution in ERP research is the **average reference**: re-reference every channel
to the mean of *all* channels simultaneously:
 
```
V_new  =  V_channel  −  mean(all channels)
```
 
This is **spatially neutral** , and the sum of all channels
is forced to zero.
 
We apply this **after ICA** so that artifact activity is not spread across channels before
the decomposition.
 
> 💡 Re-referencing is a **linear operation** — its order relative to averaging does not matter.
> *(Luck, 2014; erpinfo.org)*

### 📌 Task 2.1 – Apply average reference
 
Apply the average reference to the ICA-cleaned data (`raw_clean`).

In [ ]:
raw_clean.???(ref_channels='average')
raw_clean.???()
print('Average reference applied.')

### 📌 Task 3.2 – Butterfly plots: before vs after
 
A **butterfly plot** overlays all channels at once — great for checking signal balance.
 
**What to look for:**
- Are the traces more centred around 0 µV after re-referencing?
- Is the overall amplitude spread smaller?
- Any channel still very different from the rest? → possible bad channel for interpolation

In [ ]:
raw_before_ref = ica.apply(???, exclude=bad_components)
raw_snippet     = ???.copy().crop(tmin=???, tmax=???)
raw_ref_snippet = ???.copy().crop(tmin=???, tmax=???)

raw_snippet.plot(???=True, title='Before re-reference')
raw_ref_snippet.plot(butterfly=True, ???='After average reference')

---
## Part 3 · Bad Channel Interpolation

### What to interpolate
We now need to interpolate the channels with Bad Signals, which we marked at the start of the script, but now in our filtered, ICA-applied, re-referenced data!
first, check in that data if the channels are already marked as bad - if not look at the start of the script to see which channels you need to mark!

### How does interpolation work?
 
MNE replaces each bad channel with a **weighted average of its spatial neighbours**,
using spherical spline interpolation. For this to work, electrode positions must be set
via a **montage**.
 
```
.interpolate_bads(reset_bads=True)            # replace with neighbour average
```
 
> 💡 `reset_bads=True` clears the bads list after interpolation, so the repaired channel
> is treated as normal in all subsequent steps.
 
> 💡 How to spot bad channels: use `raw.plot()` and look for channels that are flat,
> excessively noisy, or completely different from their neighbours. In Day 2's ICA plots,
> a bad channel shows up as a **focal hotspot** dominating multiple components.


### Plot to see differences
You can use the plot() function from yesterday to take a look at the signal of the interpolated channel before and after!

---
## Part 4 · Extract Epochs
 
### What is an epoch?
 
Until now we have been working with **continuous** EEG — one long recording.
For ERP analysis we cut the continuous signal into short, fixed-length segments
called **epochs**, each time-locked to an event marker:
 
```
Continuous:  ─────────────[stim]─────────────[stim]────────────
                               │                   │
                         ┌─────┴──────┐      ┌─────┴──────┐
Epochs:                  │ -100..600ms│      │ -100..600ms│  ...
                         └────────────┘      └────────────┘
```
 
| Parameter | Value | Meaning |
|---|---|---|
| `tmin` | −0.1 s | 100 ms *before* stimulus (pre-stimulus baseline) |
| `tmax` | +0.6 s | 600 ms *after* stimulus |
| `baseline` | `None` | We apply this separately in Part 5 (more transparent) |
| `preload` | `True` | Load all epochs into RAM immediately |
| `reject` | `None` | Applied separately in Part 6 |

### 📌 Task 4.1 – Find events
 
Event markers were recorded as annotations. Convert them to an array MNE can use for epoching.

In [ ]:
events, event_id = mne.events_from_annotations(???)
print(???)
print(f'\nTotal events: {len(events)}')
print(events[:10])

### 📌 Task 4.2 – Create epochs
 
Fill in the correct event codes from the dictionary you printed above.

In [ ]:
event_id_select = {'Go': ???, 'NoGo': ???}

epochs = mne.Epochs(
    raw_clean, events, event_id=event_id_select,
    tmin=???, tmax=???,
    baseline=None, preload=True, verbose=False
)
print(epochs)
print(f'Go: {len(epochs["Go"])}, NoGo: {len(epochs["NoGo"])}')

### 📌 Task 4.3 – (Bonus) Epoch image plot
 
An **epoch image plot** stacks all individual epochs as rows (colour = amplitude).
Noisy trials stand out as bright/dark rows, and consistent ERP patterns show up
as horizontal bands of colour.

In [ ]:
# Each row = one epoch, colour = amplitude at channel Fz
epochs.plot_image(picks=['???'], title=f'{???} – Fz epoch image');

---
## Part 5 · Baseline Correction
 
### What and why?
 
EEG signals drift — the overall voltage level shifts gradually over time.
This means the starting voltage of each epoch is slightly different,
making cross-trial comparisons unfair.
 
**Baseline correction** removes this drift:
```
signal_corrected(t)  =  signal(t)  −  mean( signal[−100ms → 0ms] )
```
 
After this, every epoch starts at ≈ 0 µV at t = 0 (stimulus onset).
 
> 💡 Baseline correction is a **linear operation** — its order relative to averaging
> doesn't matter. But always apply it *before* artifact rejection, so rejection
> decisions are made on corrected data. *(Luck, 2014; erpinfo.org)*
 
> 💡 A key reason to filter the **continuous** EEG on Day 1 (rather than after epoching)
> is to prevent filter edge effects from contaminating the baseline window. *(erpinfo.org)*

### 📌 Task 5.1 – Apply baseline correction
 
The tuple is in **seconds** — not milliseconds!

In [ ]:
epochs.apply_baseline(baseline=(???, ???))

### 📌 Task 5.2 – Verify the baseline correction
 
`.get_data()` returns data in **Volts**. Multiply by `1e6` to convert to µV.

In [ ]:
baseline_check = epochs.???.crop(tmin=???, tmax=???)
mean_uv = baseline_check.???().???() * 1e6
print(f'Mean amplitude in baseline window: {mean_uv:.6f} µV  (should be ≈ 0)')

---
## Part 6 · Epoch Rejection
 
### What and why?
 
Even after ICA, some epochs may still contain large residual artifacts:
- Muscle bursts (participant coughed or moved their head)
- Electrode pops (sudden voltage jump in one channel)
- Any artifact ICA did not decompose into its own component
 
We use **threshold rejection**: discard any epoch where *any* EEG channel exceeds ±150 µV.
 
> 💡 Thresholds in MNE are always in **Volts**, not µV. So 150 µV = `150e-6` V.
 
> ⚠️ Artifact rejection is **nonlinear** — always apply it *after* re-referencing and
> baseline correction, never before. *(Luck, 2014; erpinfo.org/order-of-steps)*
 
> 💡 If one channel causes most rejections, consider **interpolating** that channel
> (replacing it with a weighted average of its neighbours) before rejecting epochs.

### 📌 Task 6.1 – Reject bad epochs

In [ ]:
reject_criteria = {'eeg': ???}

n_before = len(???)
epochs.drop_bad(reject=???)
n_after  = ???(epochs)

print(f'Epochs before: {n_before}')
print(f'Epochs after:  {n_after}')
print(f'Rejected: {n_before-n_after} ({(n_before-n_after)/n_before*100:.1f}%)')

### 📌 Task 6.2 – Plot the drop log
 
`plot_drop_log()` shows *which* epochs were dropped and *which channel* caused each rejection.
 
If one channel appears very often → consider interpolation instead of rejection.

In [ ]:
epochs.???();

---
## Part 7 · Save
 
Preprocessing is done. We save the clean epochs as a `.fif` file.
This is the output that feeds into Day 3 — where we will compute the actual ERPs,
compare Go vs. NoGo, and look at N2 and P3 components.
 
> 💡 Epoch files must end with `-epo.fif` or `_epo.fif` (MNE naming convention).

### 📌 Task 7.1 – Print summary

In [ ]:
# Summary & Save 
print('=' * 55)
print(f'  Summary for {???}')
print('=' * 55)
print(f'  Clean epochs total:     {???(epochs)}')
print(f'  Conditions:             {list(epochs.event_id.keys())}')
print(f'  Time window:            {epochs.???*1000:.0f} to {epochs.tmax*1000:.0f} ms')
print(f'  Sampling rate:          {epochs.info["???"]} Hz')
print(f'  ICA components removed: {bad_components}')
print(f'  Interpolated channels:  {list(???["bads"])}')
print('=' * 55)
for cond in epochs.event_id:
    print(f'  {cond:6s}: {len(epochs[cond])} epochs')

  Summary for B04
  Clean epochs total:     192
  Conditions:             ['Go/1', 'Go/2', 'Go/3', 'NoGo']
  Time window:            -102 to 602 ms
  Sampling rate:          256.0 Hz
  ICA components removed: [0]
  Interpolated channels:  P10
  Go/1  : 46 epochs
  Go/2  : 49 epochs
  Go/3  : 46 epochs
  NoGo  : 51 epochs


### 📌 Task 7.2 – Save the clean epochs

In [ ]:
out_path = op.join(???, 'preprocessing', f'{sub_id}_task-gonogo_epo.fif')
epochs.???(out_path, overwrite=True)
print(f'Saved: {out_path}')

---
---
# 🏆 Final Task – Loop over all participants
 
You have built a complete pipeline for **one participant**.
Now wrap everything in a **`for`-loop** so it runs for all participants automatically.
 
**Important tips:**
- Load the data **freshly inside the loop** — never reuse variables from outside
- Use a **config dictionary** per participant with bad channels and ICA components to exclude
- Use `random_state=100` in ICA so all participants are treated identically
- For large datasets, automatic `find_bads_eog()` + visual spot-checking is most practical
- Print a one-line summary per participant to track progress
 
**Bonus:** Save a rejection log CSV with number of dropped epochs and responsible channel per participant.

In [ ]:
# Per-participant settings 
participant_config = {
    'B01': {'bads': ['P10'],                                     'ica_exclude': [0, 5]},
    'B02': {'bads': ['P10', 'AF8', 'AF7', 'Fp2', 'Fp1'],        'ica_exclude': [0]},
    'B03': {'bads': ['F8', 'F10', 'AF8', 'AF7', 'Fp2', 'Fp1'],  'ica_exclude': [0]},
    'B04': {'bads': ['P10', 'AF4', 'AF7', 'Fp2', 'Fp1'],                       'ica_exclude': [0]},
}

event_id_select = {'Go/1': 1, 'Go/2': 2, 'Go/3': 3, 'NoGo': 4}

for sub_id, config in participant_config.items():
    print(f'\n── Processing {sub_id} ──────────────────────────────')

    # Load preprocessed (filtered + resampled) data from Day 1
    raw_filt_path = op.join(data_path, 'preprocessing', f'{sub_id}_filtered_raw.fif')
    raw = mne.io.read_raw_fif(raw_filt_path, preload=True)

    # Set montage & interpolate participant-specific bad channels BEFORE ICA
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    raw.info['bads'] = config['bads']
    raw.interpolate_bads(reset_bads=True)
    print(f'  Interpolated: {config["bads"]}')

    # Create 1 Hz high-pass copy for ICA fitting (same data chain)
    raw_ica = raw.copy().filter(l_freq=1.0, h_freq=None)

    # Fit ICA on the 1 Hz copy
    ica = ICA(n_components=15, max_iter='auto', random_state=100)
    ica.fit(raw_ica)

    # Remove artifact components — inspected visually per participant!
    bad_components = config['ica_exclude']
    raw_clean = ica.apply(raw.copy(), exclude=bad_components)
    print(f'  ICA excluded: {bad_components}')

    # Apply average reference AFTER ICA
    raw_clean.set_eeg_reference(ref_channels='average')
    raw_clean.apply_proj()

    # Epoch, baseline correct, reject
    events, _ = mne.events_from_annotations(raw_clean)
    epochs = mne.Epochs(
        raw_clean, events, event_id=event_id_select,
        tmin=-0.1, tmax=0.6,
        baseline=None, preload=True, verbose=False
    )
    epochs.apply_baseline(baseline=(-0.1, 0))
    epochs.drop_bad(reject={'eeg': 200e-6})

    out_path = op.join(data_path, 'preprocessing', f'{sub_id}_task-gonogo_epo.fif')
    epochs.save(out_path, overwrite=True)
    print(f'  Done: {len(epochs)} clean epochs → {out_path}')

print('\nAll participants processed!')

---
**Good luck!**

*References: MNE ERP tutorial (mne.tools/stable/auto_tutorials/evoked/30_eeg_erp.html) · Luck (2014) via erpinfo.org/order-of-steps · ICLabel taxonomy via labeling.ucsd.edu/tutorial/labels*